In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pickle
from sklearn.metrics import roc_auc_score


In [4]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
statlog_german_credit_data = fetch_ucirepo(id=144) 
  
# data (as pandas dataframes) 
X = statlog_german_credit_data.data.features 
y = statlog_german_credit_data.data.targets 

In [5]:
df_credit = pd.concat([X, y], axis=1)
df_credit.columns=["status", "duration", "credit_history", "purpose", "amount", 
                "savings", "employment_duration", "installment_rate",
                "sex", "other_debtors",
                "present_residence", "property",
                "age", "other_installment_plans",
                "housing", "number_credits",
                "job", "people_liable", "telephone", "foreign_worker",
                "target_credit"]
df_credit.dropna(inplace=True)

# Remap target_credit: 1 (good) -> 0, 2 (bad) -> 1
df_credit['target_credit'] = df_credit['target_credit'].map({1: 0, 2: 1})

# Convert categorical string codes to integers
# Status: A11=1, A12=2, A13=3, A14=4
status_map = {'A11': 1, 'A12': 2, 'A13': 3, 'A14': 4}
# Credit history: A30=1, A31=2, A32=3, A33=4, A34=5
credit_history_map = {'A30': 1, 'A31': 2, 'A32': 3, 'A33': 4, 'A34': 5}
# Purpose: A40=1, A41=2, A42=3, A43=4, A44=5, A45=6, A46=7, A48=8, A49=9, A410=10
purpose_map = {'A40': 1, 'A41': 2, 'A42': 3, 'A43': 4, 'A44': 5, 'A45': 6, 'A46': 7, 'A48': 8, 'A49': 9, 'A410': 10}
# Savings: A61=1, A62=2, A63=3, A64=4, A65=5
savings_map = {'A61': 1, 'A62': 2, 'A63': 3, 'A64': 4, 'A65': 5}
# Employment duration: A71=1, A72=2, A73=3, A74=4, A75=5
employment_map = {'A71': 1, 'A72': 2, 'A73': 3, 'A74': 4, 'A75': 5}
# Personal status/sex: A91=male, A92=female, A93=male, A94=male, A95=female
personal_status_map = {'A91': 0, 'A92': 1, 'A93': 0, 'A94': 0, 'A95': 1}
# Other debtors: A101=1, A102=2, A103=3
other_debtors_map = {'A101': 1, 'A102': 2, 'A103': 3}
# Property: A121=1, A122=2, A123=3, A124=4
property_map = {'A121': 1, 'A122': 2, 'A123': 3, 'A124': 4}
# Other installment plans: A141=1, A142=2, A143=3
installment_plans_map = {'A141': 1, 'A142': 2, 'A143': 3}
# Housing: A151=1, A152=2, A153=3
housing_map = {'A151': 1, 'A152': 2, 'A153': 3}
# Job: A171=1, A172=2, A173=3, A174=4
job_map = {'A171': 1, 'A172': 2, 'A173': 3, 'A174': 4}
# Telephone: A191=1, A192=2
telephone_map = {'A191': 1, 'A192': 2}
# Foreign worker: A201=1, A202=2
foreign_worker_map = {'A201': 1, 'A202': 2}

# Apply mappings
df_credit['status'] = df_credit['status'].map(status_map)
df_credit['credit_history'] = df_credit['credit_history'].map(credit_history_map)
df_credit['purpose'] = df_credit['purpose'].map(purpose_map)
df_credit['savings'] = df_credit['savings'].map(savings_map)
df_credit['employment_duration'] = df_credit['employment_duration'].map(employment_map)
df_credit['sex'] = df_credit['sex'].map(personal_status_map)
df_credit['other_debtors'] = df_credit['other_debtors'].map(other_debtors_map)
df_credit['property'] = df_credit['property'].map(property_map)
df_credit['other_installment_plans'] = df_credit['other_installment_plans'].map(installment_plans_map)
df_credit['housing'] = df_credit['housing'].map(housing_map)
df_credit['job'] = df_credit['job'].map(job_map)
df_credit['telephone'] = df_credit['telephone'].map(telephone_map)
df_credit['foreign_worker'] = df_credit['foreign_worker'].map(foreign_worker_map)

# Ensure all values are numeric
df_credit = df_credit.astype(int)

In [14]:
df_credit.shape

(1000, 21)

In [6]:
# define protected attributes 
protected_attributes = ["age", "sex", "foreign_worker"]

In [7]:
# feature categorization including all features
numerical_features = ["duration", "amount", "installment_rate", "present_residence", "number_credits", "people_liable", "age"]
ordinal_features = ["status", "credit_history", "savings", "employment_duration", "other_installment_plans", "housing", "job", "telephone"]
nominal_features = ["purpose", "other_debtors", "property", "sex", "foreign_worker"]

# Define categorical features (ordinal + nominal)
categorical_features = ordinal_features + nominal_features

# Create feature descriptions 
feature_descriptions = [
    "Status of checking account (1: <0 DM, 2: 0-200 DM, 3: ≥200 DM/salary, 4: none)",
    "Duration of credit request in months",
    "Credit history rating (1: no credits-all paid, 2: all paid at bank, 3: existing paid till now, 4: past delays, 5: critical/other credits)",
    "Purpose of credit (1: new car, 2: used car, 3: furniture/equipment, 4: radio/TV, 5: domestic appliances, 6: repairs, 7: education, 8: retraining, 9: business, 10: others)",
    "Amount of credit requested in DM",
    "Savings account status (1: <100 DM, 2: 100-500 DM, 3: 500-1000 DM, 4: ≥1000 DM, 5: unknown/none)",
    "Employment duration (1: unemployed, 2: <1 year, 3: 1-4 years, 4: 4-7 years, 5: ≥7 years)",
    "Installment rate as percentage of disposable income",
    "Sex (0: male, 1: female)",
    "Other debtors/guarantors (1: none, 2: co-applicant, 3: guarantor)",
    "Years at present residence",
    "Property status (1: real estate, 2: building society savings/life insurance, 3: car/other, 4: unknown/none)",
    "Age in years",
    "Other installment plans (1: bank, 2: stores, 3: none)",
    "Housing situation (1: rent, 2: own, 3: for free)",
    "Number of existing credits at this bank",
    "Job type (1: unemployed/unskilled-non-resident, 2: unskilled-resident, 3: skilled/official, 4: management/self-employed/highly skilled)",
    "Number of people financially dependent",
    "Telephone (1: none, 2: yes registered)",
    "Foreign worker (1: yes, 2: no)"
]


In [8]:
random_state = 1234
credit_train, credit_test = train_test_split(df_credit, test_size=0.2, random_state=random_state)

credit_train.to_parquet("train_cleaned.parquet")
credit_test.to_parquet("test_cleaned.parquet")

# Separate features and target (keep all features including protected attributes)
x_train = credit_train.drop(columns=["target_credit"])
y_train = credit_train["target_credit"]
x_test = credit_test.drop(columns=["target_credit"])
y_test = credit_test["target_credit"]

In [9]:
# random forest model WITHOUT protected attributes 

# Create training and test sets without protected attributes for the model
x_train_model = x_train.drop(columns=protected_attributes)
x_test_model = x_test.drop(columns=protected_attributes)

rf_model = RandomForestClassifier(random_state=random_state, class_weight='balanced')
rf_model.fit(x_train_model, y_train)

# evaluate the model on accuracy and AUC metrics
rf_pred = rf_model.predict(x_test_model)
rf_pred_proba = rf_model.predict_proba(x_test_model)[:, 1]  # Get probability for class 1
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"Accuracy: {rf_accuracy * 100:.2f}%")
rf_auc = roc_auc_score(y_test, rf_pred_proba)
print(f"AUC: {rf_auc * 100:.2f}%")

# Save the model
with open('RF.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

Accuracy: 74.50%
AUC: 77.78%


In [10]:
# Define feature value mappings for readable display
FEATURE_MAPPINGS = {
    'status': {1: '<0 DM', 2: '0-200 DM', 3: '≥200 DM/salary', 4: 'none'},
    'credit_history': {1: 'no credits/all paid', 2: 'all paid at bank', 3: 'existing paid', 4: 'past delays', 5: 'critical/other credits'},
    'purpose': {1: 'new car', 2: 'used car', 3: 'furniture/equipment', 4: 'radio/TV', 5: 'domestic appliances', 6: 'repairs', 7: 'education', 8: 'retraining', 9: 'business', 10: 'others'},
    'savings': {1: '<100 DM', 2: '100-500 DM', 3: '500-1000 DM', 4: '≥1000 DM', 5: 'unknown/none'},
    'employment_duration': {1: 'unemployed', 2: '<1 year', 3: '1-4 years', 4: '4-7 years', 5: '≥7 years'},
    'other_debtors': {1: 'none', 2: 'co-applicant', 3: 'guarantor'},
    'property': {1: 'real estate', 2: 'building society/life insurance', 3: 'car/other', 4: 'unknown/none'},
    'other_installment_plans': {1: 'bank', 2: 'stores', 3: 'none'},
    'sex': {0: 'male', 1: 'female'},
    'housing': {1: 'rent', 2: 'own', 3: 'for free'},
    'job': {1: 'unemployed/unskilled-non-resident', 2: 'unskilled-resident', 3: 'skilled/official', 4: 'management/self-employed/highly skilled'},
    'telephone': {1: 'none', 2: 'yes registered'},
    'foreign_worker': {1: 'yes', 2: 'no'}
}

def map_value(feature_name, value):
    """Map encoded values to readable names"""
    if feature_name in FEATURE_MAPPINGS:
        mapping = FEATURE_MAPPINGS[feature_name]
        if value in mapping:
            return mapping[value]
        if isinstance(value, float):
            if int(value) in mapping:
                return mapping[int(value)]
    return str(value)

feature_descriptions_simple = [
    "Status of checking account",
    "Duration of credit request in months",
    "Credit history rating",
    "Purpose of credit",
    "Amount of credit requested in DM",
    "Savings account status",
    "Employment duration",
    "Installment rate as percentage of disposable income",
    "The sex of the applicant",
    "Other debtors/guarantors",
    "Years at present residence",
    "Property status",
    "The age of the applicant in years",
    "Other installment plans",
    "Housing situation",
    "Number of existing credits at this bank",
    "Job type",
    "Number of people financially dependent",
    "has a telephone or not",
    "Foreign worker or not"
]

# Build feature dataframe with 
feature_data = []
for i, col in enumerate(x_train.columns):
    row = {
        "feature_name": col,
        "feature_desc": feature_descriptions_simple[i]
    }
    feature_data.append(row)

feature_desc_df = pd.DataFrame(feature_data)

dataset_description = "The dataset contains information from the 1970s in Germany on a series of debtors that took a loan from the bank. It includes detailed categorical and numerical variables about their financial situation. Protected attributes age and sex were not used to make the machine prediction. Keep in mind that at the time Germany used Deutsche Marks (DM) with an average yearly salary of 10,000 to 20,000 DM"
target_description = "The target variable indicates whether the customer is a bad credit risk (1) or a good credit (0). We are predicting the bad credit class."
task_description = "The ML model aims to predict whether a new customer will be a bad credit risk"

# Dataset info for original features (x_train_no_protected)
dataset_info = {
    "dataset_description": dataset_description,
    "target_description": target_description,
    "task_description": task_description,
    "feature_description": feature_desc_df
}

# Save
with open('dataset_info', 'wb') as f:
    pickle.dump(dataset_info, f)

In [12]:
dataset_info

{'dataset_description': 'The dataset contains information from the 1970s in Germany on a series of debtors that took a loan from the bank. It includes detailed categorical and numerical variables about their financial situation. Protected attributes age and sex were not used to make the machine prediction. Keep in mind that at the time Germany used Deutsche Marks (DM) with an average yearly salary of 10,000 to 20,000 DM',
 'target_description': 'The target variable indicates whether the customer is a bad credit risk (1) or a good credit (0). We are predicting the bad credit class.',
 'task_description': 'The ML model aims to predict whether a new customer will be a bad credit risk',
 'feature_description':                feature_name                                       feature_desc
 0                    status                         Status of checking account
 1                  duration               Duration of credit request in months
 2            credit_history                 